In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [3]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

12.15

In [ ]:
def calculate_heikin_ashi(df):
    ha_close = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    ha_open = (ha_close.shift(1) + ha_close.shift(1)) / 2
    ha_high = df[['high', 'open', 'close']].max(axis=1)
    ha_low = df[['low', 'open', 'close']].min(axis=1)

    return pd.DataFrame({'ha_open': ha_open, 'ha_high': ha_high, 'ha_low': ha_low, 'ha_close': ha_close, 'time':df.time})

In [ ]:
import numpy as np
import pandas as pd
import pandas_ta as pdt

def supertrend(factor, atr_length, high, low, close):
    atr = pdt.atr(high, low, close, atr_length)
    basic_upper_band = (high + low) / 2 + factor * atr
    basic_lower_band = (high + low) / 2 - factor * atr
    bullish_signal = close > basic_upper_band
    bearish_signal = close < basic_lower_band
    bullish_supertrend = np.full_like(close, np.nan)
    bearish_supertrend = np.full_like(close, np.nan)

    for i in range(1, len(close)):
        if bullish_signal[i] or (bullish_supertrend[i-1] and close[i-1] > basic_upper_band[i-1]):
            bullish_supertrend[i] = max(basic_upper_band[i], bullish_supertrend[i-1])
        else:
            bullish_supertrend[i] = basic_upper_band[i]

        if bearish_signal[i] or (bearish_supertrend[i-1] and close[i-1] < basic_lower_band[i-1]):
            bearish_supertrend[i] = min(basic_lower_band[i], bearish_supertrend[i-1])
        else:
            bearish_supertrend[i] = basic_lower_band[i]

    direction = np.where(close > bullish_supertrend, 1, np.where(close < bearish_supertrend, -1, 0))
    supertrend = np.where(direction == 1, bullish_supertrend, bearish_supertrend)
    
    return supertrend, direction

# Example usage:
# Assuming df is your DataFrame containing OHLC data
# Replace this with your actual DataFrame
# Example:
# df = pd.DataFrame({'open': [...], 'high': [...], 'low': [...], 'close': [...]})

# Convert input parameters from Pine Script to Python
# factor = 3.0
# atr_length = 10




In [ ]:
pdt.atr?

In [163]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [164]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertren
    return rates_frame

In [17]:
a = get_values('GBPUSD', 1000, 50, 'D1')

In [18]:
a

,open,high,low,close,sma
time,,,,,
2020-11-13,1.31132,1.32001,1.31091,1.31956,1.296331
2020-11-16,1.31710,1.32420,1.31650,1.32000,1.296771
2020-11-17,1.31863,1.32724,1.31857,1.32488,1.297269
2020-11-18,1.32434,1.33117,1.32422,1.32696,1.298204
2020-11-19,1.32691,1.32789,1.31953,1.32556,1.299128
...,...,...,...,...,...
2024-06-18,1.27032,1.27204,1.26683,1.27077,1.261365
2024-06-19,1.27043,1.27394,1.26997,1.27176,1.261721
2024-06-20,1.27177,1.27230,1.26542,1.26569,1.261930


In [ ]:
type(direction)

In [ ]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [ ]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 25, 'H4')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=1
            
        if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 25, 'H4')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=1
            
        if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [165]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "GBPUSD"
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
a = get_values(symbol, 10000, 50, 'M30')
lot = 0.05
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

# 
print("here")
print(lot)
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].close <= a.iloc[i-1].sma and  a.iloc[i-1].open >= a.iloc[i-1].sma:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=1
            
#         if a.iloc[i-1].close >= a.iloc[i-1].sma and a.iloc[i-1].open <= a.iloc[i-1].sma:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if a.iloc[i].high >= a.iloc[i].sma and  (a.iloc[i].high - a.iloc[i].sma) >= 0.00300:
            pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma+0.00300, mt5.ORDER_TYPE_SELL)
            print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
            if pp1<=-10:
                profit.append(-10)
            else:
                profit.append(pp1)
            check=0
#         elif pp <= -300:
#             profit.append(-300)
#             check=0
        elif pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)



#     if check==2:
#         sell_price = a.iloc[i].close
#         pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
#         print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#         if pp >= 0.0:
# #             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
# #         elif a.iloc[i].close > a.iloc[i].ema:
# #             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
# #             profit.append(pp)
#         if pp<-10:
#             profit.append(-10)
#         else:
#             profit.append(pp)
#         check=0

here
0.05
2023-09-12 00:00:00
-4.55--1.2509594---1.25123--1.25082--2023-09-12 00:00:00
-3.7--1.2510554---1.25106--1.25082--2023-09-12 00:30:00
-3.6--1.2511136---1.25104--1.25082--2023-09-12 01:00:00
-4.05--1.2511848---1.25113--1.25082--2023-09-12 01:30:00
-4.0--1.251253---1.25112--1.25082--2023-09-12 02:00:00
-2.1--1.2513114---1.25074--1.25082--2023-09-12 02:30:00
-0.20000000000000018--1.2513588---1.2503600000000001--1.25082--2023-09-12 03:00:00
-1.2--1.2513944000000001---1.2505600000000001--1.25082--2023-09-12 03:30:00
-0.8--1.251436---1.25048--1.25082--2023-09-12 04:00:00
-1.05--1.2514748---1.25053--1.25082--2023-09-12 04:30:00
-2.35--1.251516---1.25079--1.25082--2023-09-12 05:00:00
-6.15--1.2515572---1.25155--1.25082--2023-09-12 05:30:00
-5.1--1.2515863999999999---1.25134--1.25082--2023-09-12 06:00:00
-6.55--1.2516192000000002---1.25163--1.25082--2023-09-12 06:30:00
-3.55--1.2516436---1.25103--1.25082--2023-09-12 07:00:00
-3.55--1.2516628---1.25103--1.25082--2023-09-12 07:30:00
-5.3

2023-09-28 05:30:00
-0.5--1.2139342---1.21306--1.21346--2023-09-28 05:30:00
-2.65--1.2139178---1.21349--1.21346--2023-09-28 06:00:00
-3.45--1.2138984000000002---1.21365--1.21346--2023-09-28 06:30:00
-1.6--1.2138756---1.21328--1.21346--2023-09-28 07:00:00
-5.05--1.2138730000000002---1.21397--1.21346--2023-09-28 07:30:00
-7.1--1.2138778000000001---1.21438--1.21346--2023-09-28 08:00:00
-7.5--1.2138742---1.21446--1.21346--2023-09-28 08:30:00
-2.15--1.2138537999999999---1.21339--1.21346--2023-09-28 09:00:00
1.9500000000000002--1.2138164---1.21257--1.21346--2023-09-28 09:30:00
2023-09-29 17:00:00
13.899999999999999--1.2219023999999998---1.21831--1.22159--2023-09-29 17:00:00
2023-10-02 11:00:00
-0.1499999999999999--1.2208558---1.21887--1.21934--2023-10-02 11:00:00
0.9500000000000002--1.2207442---1.21865--1.21934--2023-10-02 11:30:00
2023-10-03 17:30:00
-3.1--1.2086891999999998---1.20782--1.2077--2023-10-03 17:30:00
2.7--1.2085484000000002---1.20666--1.2077--2023-10-03 18:00:00
2023-10-03 20:0

2023-11-15 14:30:00
-2.0--1.2467242---1.24611--1.24621--2023-11-15 14:30:00
-3.85--1.2470678000000002---1.24648--1.24621--2023-11-15 15:00:00
7.5--1.247366---1.24421--1.24621--2023-11-15 15:30:00
2023-11-16 22:30:00
-3.15--1.2407496---1.24081--1.24068--2023-11-16 22:30:00
-4.5--1.2407523999999999---1.24108--1.24068--2023-11-16 23:00:00
-5.65--1.2407602000000002---1.24131--1.24068--2023-11-16 23:30:00
-5.15--1.2407595999999999---1.24121--1.24068--2023-11-17 00:00:00
-6.2--1.240759---1.24142--1.24068--2023-11-17 00:30:00
-6.95--1.2407626---1.24157--1.24068--2023-11-17 01:00:00
-5.95--1.2407564---1.24137--1.24068--2023-11-17 01:30:00
-4.95--1.2407478---1.2411699999999999--1.24068--2023-11-17 02:00:00
-7.9--1.2407408---1.24176--1.24068--2023-11-17 02:30:00
-7.45--1.240731---1.24167--1.24068--2023-11-17 03:00:00
-5.45--1.2407204---1.24127--1.24068--2023-11-17 03:30:00
-4.0--1.2407228---1.24098--1.24068--2023-11-17 04:00:00
-3.9--1.2407364---1.24096--1.24068--2023-11-17 04:30:00
-3.55--1.240

-1.9--1.263391---1.26311--1.26323--2023-12-05 13:00:00
2.75--1.2633024---1.26218--1.26323--2023-12-05 13:30:00
2023-12-05 16:30:00
3.4000000000000004--1.262678---1.26085--1.26203--2023-12-05 16:30:00
2023-12-06 08:30:00
0.20000000000000018--1.2608084---1.25968--1.26022--2023-12-06 08:30:00
2023-12-06 10:30:00
1.7000000000000002--1.2606222---1.25933--1.26017--2023-12-06 10:30:00
2023-12-06 17:00:00
0.10000000000000009--1.2597838000000001---1.25879--1.25931--2023-12-06 17:00:00
2023-12-07 11:00:00
-1.05--1.2572059999999998---1.25663--1.25692--2023-12-07 11:00:00
-10.7--1.2571906---1.2585600000000001--1.25692--2023-12-07 11:30:00
-5.85--1.2571326---1.25759--1.25692--2023-12-07 12:00:00
-11.5--1.2571082---1.25872--1.25692--2023-12-07 12:30:00
-8.9--1.2570734000000001---1.2582--1.25692--2023-12-07 13:00:00
-3.0--1.257013---1.25702--1.25692--2023-12-07 13:30:00
0.25--1.2569642---1.25637--1.25692--2023-12-07 14:00:00
2023-12-07 14:30:00
-0.5--1.2569098---1.25596--1.25636--2023-12-07 14:30:00


-23.2--1.2715682---1.2732--1.26906--2023-12-27 12:30:00
-22.2--1.271644---1.2730000000000001--1.26906--2023-12-27 13:00:00
-24.6--1.2717178---1.27348--1.26906--2023-12-27 13:30:00
-26.25--1.2718064---1.27381--1.26906--2023-12-27 14:00:00
-28.25--1.2719040000000001---1.27421--1.26906--2023-12-27 14:30:00
-35.5--1.2720352---1.27566--1.26906--2023-12-27 15:00:00
PP1 ---> -29.88--1.2720352---1.27566-2023-12-27 15:00:00
2023-12-29 10:00:00
-5.5--1.2753634---1.27522--1.27462--2023-12-29 10:00:00
1.4500000000000002--1.2752042000000001---1.27383--1.27462--2023-12-29 10:30:00
2023-12-29 11:00:00
6.15--1.2750186---1.2721--1.27383--2023-12-29 11:00:00
2023-12-29 15:00:00
-6.85--1.2739398---1.27275--1.27188--2023-12-29 15:00:00
-3.15--1.2738506---1.2720099999999999--1.27188--2023-12-29 15:30:00
2.45--1.2737608---1.27089--1.27188--2023-12-29 16:00:00
2023-12-29 18:30:00
-4.5--1.2735750000000001---1.27402--1.27362--2023-12-29 18:30:00
-9.6--1.273596---1.27504--1.27362--2023-12-29 19:00:00
-15.4--1.2

...

2024-04-03 10:30:00
-2.05--1.2570104---1.25654--1.25663--2024-04-03 10:30:00
-6.6--1.257056---1.25745--1.25663--2024-04-03 11:00:00
-7.65--1.2571024---1.25766--1.25663--2024-04-03 11:30:00
-9.2--1.2571484---1.25797--1.25663--2024-04-03 12:00:00
-5.4--1.2571696---1.25721--1.25663--2024-04-03 12:30:00
-3.45--1.257168---1.25682--1.25663--2024-04-03 13:00:00
-7.6--1.2571838---1.25765--1.25663--2024-04-03 13:30:00
-9.85--1.257215---1.2581--1.25663--2024-04-03 14:00:00
-8.0--1.2572358---1.25773--1.25663--2024-04-03 14:30:00
-7.0--1.2572581999999999---1.25753--1.25663--2024-04-03 15:00:00
-5.55--1.257257---1.25724--1.25663--2024-04-03 15:30:00
-5.9--1.2572748---1.25731--1.25663--2024-04-03 16:00:00
-5.9--1.2573084---1.25731--1.25663--2024-04-03 16:30:00
-22.45--1.2574007999999999---1.26062--1.25663--2024-04-03 17:00:00
PP1 ---> -18.85--1.2574007999999999---1.26062-2024-04-03 17:00:00
2024-04-04 21:30:00
-0.6499999999999999--1.2656964000000002---1.2650299999999999--1.2654--2024-04-04 21:30:00


-9.95--1.2551186---1.25488--1.25339--2024-05-06 09:00:00
-13.1--1.2551298---1.2555100000000001--1.25339--2024-05-06 09:30:00
-20.05--1.2551716---1.2569--1.25339--2024-05-06 10:00:00
-19.4--1.2552194---1.25677--1.25339--2024-05-06 10:30:00
-24.05--1.2552856---1.2577--1.25339--2024-05-06 11:00:00
-25.85--1.255352---1.25806--1.25339--2024-05-06 11:30:00
-23.45--1.2553874---1.25758--1.25339--2024-05-06 12:00:00
-24.85--1.2554266---1.25786--1.25339--2024-05-06 12:30:00
-24.1--1.2554568---1.2577099999999999--1.25339--2024-05-06 13:00:00
-23.0--1.2554876---1.25749--1.25339--2024-05-06 13:30:00
-23.6--1.2555288---1.2576100000000001--1.25339--2024-05-06 14:00:00
-23.4--1.2555644000000001---1.25757--1.25339--2024-05-06 14:30:00
-28.15--1.255625---1.25852--1.25339--2024-05-06 15:00:00
PP1 ---> -26.18--1.255625---1.25852-2024-05-06 15:00:00
2024-05-06 22:30:00
-2.9--1.2561172---1.25599--1.25591--2024-05-06 22:30:00
-2.95--1.2561306---1.256--1.25591--2024-05-06 23:00:00
-3.85--1.2561574---1.25618--

-5.4--1.2721914---1.2718--1.27122--2024-05-23 00:00:00
-5.3--1.2722108---1.2717800000000001--1.27122--2024-05-23 00:30:00
-5.2--1.2722310000000001---1.27176--1.27122--2024-05-23 01:00:00
-6.45--1.2722502---1.2720099999999999--1.27122--2024-05-23 01:30:00
-6.5--1.27227---1.27202--1.27122--2024-05-23 02:00:00
-6.3--1.2722888---1.27198--1.27122--2024-05-23 02:30:00
-7.4--1.2723066---1.2722--1.27122--2024-05-23 03:00:00
-6.1--1.2723268---1.27194--1.27122--2024-05-23 03:30:00
-6.1--1.2723381999999999---1.27194--1.27122--2024-05-23 04:00:00
-7.65--1.2723572---1.27225--1.27122--2024-05-23 04:30:00
-6.6--1.2723758---1.27204--1.27122--2024-05-23 05:00:00
-8.4--1.2723902---1.2724--1.27122--2024-05-23 05:30:00
-9.0--1.2724036---1.27252--1.27122--2024-05-23 06:00:00
-10.55--1.2724276---1.27283--1.27122--2024-05-23 06:30:00
-10.05--1.2724576---1.27273--1.27122--2024-05-23 07:00:00
-9.0--1.2724836000000002---1.27252--1.27122--2024-05-23 07:30:00
-8.95--1.272511---1.27251--1.27122--2024-05-23 08:00:0

2024-06-10 16:00:00
0.20000000000000018--1.271864---1.27066--1.2711999999999999--2024-06-10 16:00:00
2024-06-11 10:00:00
-1.0--1.2722934---1.27164--1.27194--2024-06-11 10:00:00
-3.9--1.2723170000000001---1.27222--1.27194--2024-06-11 10:30:00
-6.45--1.2723422---1.27273--1.27194--2024-06-11 11:00:00
-4.9--1.2723456---1.2724199999999999--1.27194--2024-06-11 11:30:00
-4.55--1.2723482---1.2723499999999999--1.27194--2024-06-11 12:00:00
-9.45--1.2724004---1.27333--1.27194--2024-06-11 12:30:00
-14.65--1.2724824---1.27437--1.27194--2024-06-11 13:00:00
-13.75--1.2725718000000001---1.27419--1.27194--2024-06-11 13:30:00
-12.55--1.2726704---1.2739500000000001--1.27194--2024-06-11 14:00:00
-11.55--1.272754---1.27375--1.27194--2024-06-11 14:30:00
-10.75--1.2728106---1.27359--1.27194--2024-06-11 15:00:00
-14.6--1.2728698---1.27436--1.27194--2024-06-11 15:30:00
-8.3--1.2728774---1.2731--1.27194--2024-06-11 16:00:00
0.6499999999999999--1.2728793999999999---1.27131--1.27194--2024-06-11 16:30:00
2024-06-1

-9.55--1.2632776---1.26468--1.26327--2024-06-27 12:30:00
-10.65--1.2632408000000002---1.2649--1.26327--2024-06-27 13:00:00
-11.7--1.2632067999999999---1.26511--1.26327--2024-06-27 13:30:00
-10.2--1.2631902---1.26481--1.26327--2024-06-27 14:00:00
-6.5--1.2631598000000002---1.26407--1.26327--2024-06-27 14:30:00
-9.15--1.263172---1.2646--1.26327--2024-06-27 15:00:00
-13.5--1.2631632---1.26547--1.26327--2024-06-27 15:30:00
-15.85--1.2631736---1.26594--1.26327--2024-06-27 16:00:00
-17.3--1.2632092---1.26623--1.26327--2024-06-27 16:30:00
PP1 ---> -14.7--1.2632092---1.26623-2024-06-27 16:30:00
2024-06-28 02:00:00
-3.55--1.2639269999999998---1.26407--1.26386--2024-06-28 02:00:00
-5.05--1.2639740000000002---1.26437--1.26386--2024-06-28 02:30:00
-5.05--1.2640190000000002---1.26437--1.26386--2024-06-28 03:00:00
2.6500000000000004--1.264035---1.2628300000000001--1.26386--2024-06-28 03:30:00
2024-06-28 04:00:00
0.10000000000000009--1.264048---1.26231--1.2628300000000001--2024-06-28 04:00:00
2024-06

In [166]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

120.60000000000008
Total negative sm -->-460
Total negative -->46
Total positive sm -->580.5999999999997
Total positive -->180
Length 226


In [ ]:
profit.sort()

In [ ]:
profit

In [ ]:
a.iloc[-2].name == a.iloc[-2].name

In [ ]:
symbol = "BTCUSD"
a = get_values(symbol, 20000, 150, 'M15')

In [ ]:
a.iloc[-63]

In [ ]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M10':mt5.TIMEFRAME_M10, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)

    rates_frame = calculate_heikin_ashi(rates_frame)
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['sma'] = rates_frame['close'].rolling(window=50).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]

    return rates_frame

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 150, 'M10')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].ha_open < a.iloc[j].ha_close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].ha_close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].ha_open
            check=1
            
        if a.iloc[i-1].ha_close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].ha_open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].ha_close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].ha_close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].ha_close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].ha_close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [161]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 50)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
    rates_frame['sma']= rates_frame['close'].rolling(window=50).mean()

    return rates_frame


def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)


def run(symbol):
    check = 0
    lot = 0.02
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if True:
            
            order_time = datetime.fromtimestamp(time.time(), tz= pytz.timezone('Etc/GMT-3'))
#  order_time.hour <= 1 and 
            if a.iloc[-2].close <= a.iloc[-2].sma and a.iloc[-3].close >= a.iloc[-2].close and sell_check == 0:   

                result_sell = Action(symbol, lot, sell)
                print(f"Symbol-->{symbol} ||| Type-->Sell  ||| Ticket_No-->{result_sell.order}")

                sell_check = 1

                buy_check = 0
                hour_passed = False

            if a.iloc[-2].close > a.iloc[-2].sma and a.iloc[-3].close < a.iloc[-2].close and buy_check == 0:

                result_buy = Action(symbol, lot, buy)
                print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                buy_check = 1

                sell_check = 0

                hour_passed = False

            #############################################################

        t = datetime.fromtimestamp(time.time(), tz= pytz.timezone('Etc/GMT-3'))
        if sell_check == 1:
            result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close
            
            if result_sell.comment == "Requote":
                result_sell = Action_close(result_sell.order, symbol, sell, lot)
                print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
            else:
                print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
            sell_check = 0

            hour_passed = True


        if buy_check == 1:
            result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

            if result_buy.comment == "Requote":
                result_buy = Action_close(result_buy.order, symbol, buy, lot)
                print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
            else:
                print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
            buy_check = 0

            hour_passed = True
        time.sleep(1)


#         print(f"Sleep Time-->{((60*60 - t.minute*60) + 1)}")
#         time.sleep((60*60 - t.minute*60) + 1)

for symbol in ['GBPUSD']:
    run(symbol)

GBPUSD


KeyboardInterrupt: 